# 00 — Run control and progress

**ITRI626 · Colile Sibanda**

The control panel for the training runs. Use it to see what is done, what is
running on another machine, and what this machine should start next; to launch
or resume a run; and to watch a run's progress while it trains.

It reads the same files as `run_status.py`, so it agrees with the terminal and
with the other machines. Nothing here edits a result: the cells that launch
training call the same orchestrators the shell scripts do.

**Before anything else, `git pull`** — claims and finished metrics arrive that way.

In [ ]:
# The same opening lines as every other notebook: path, imports, display width.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"Project root: {ROOT}")

## 1 · Where everything stands

`NOT STARTED` is free to claim · `RUNNING` belongs to another machine · `PAUSED`
can be resumed anywhere · `NEEDS EVAL` has finished training · `COMPLETE` is done.

In [ ]:
from run_status import MODELS, progress, state_of


def status_frame() -> pd.DataFrame:
    """Every model's state, as run_status.py reports it on the command line."""
    rows = []
    for name, spec in MODELS.items():
        row = progress(name, spec)
        claim = row["claim"] or {}
        rows.append(
            {
                "model": name,
                "family": row["family"],
                "state": state_of(row),
                "epochs": f"{row['epochs_done']}/{row['epochs_target']}",
                "best val macro-F1": row["best_val_macro_f1"],
                "test macro-F1": row["test_macro_f1"],
                "machine": claim.get("machine", ""),
                "claim age (min)": claim.get("age_minutes", ""),
            }
        )
    return pd.DataFrame(rows)


status_frame()

## 2 · What this machine should do next

Re-run the cell above and this one after a `git pull` to see the current answer.

In [ ]:
free = [
    name
    for name, spec in MODELS.items()
    if state_of(progress(name, spec)) in ("NOT STARTED", "PAUSED")
]
needs_eval = [
    name
    for name, spec in MODELS.items()
    if state_of(progress(name, spec)) == "NEEDS EVAL"
]

if needs_eval:
    name = needs_eval[0]
    print(f"Evaluate {name} first — it is trained but has no held-out metrics:")
    print(
        f"  python run_evaluate.py --config {MODELS[name]['config']} "
        f"--env local --model {name}"
    )
elif free:
    name = free[0]
    print(f"Free to start here: {name}")
    print(f"  ./train_model.sh {name}")
    print()
    print("Or from this notebook, with the cell in section 3.")
else:
    print("Every model is trained and evaluated. Go to section 5, the comparison.")

## 3 · Start or resume a run from here

This calls the same `run_train.py` the shell script does, so a run started in
this notebook is resumable from the terminal, and a run started in the terminal
is resumable here.

Training runs for hours and the notebook is blocked while it does. For a long
run prefer the terminal (`./train_model.sh <model>`), which also keeps the claim
alive and evaluates when training finishes.

Set `MODEL`, then remove the `raise` line to run it.

In [ ]:
MODEL = "resnet50"   # baseline_cnn_v4 | resnet50 | vit_b16

raise SystemExit("Remove this line to start the run.")

import subprocess

spec = MODELS[MODEL]
state = state_of(progress(MODEL, spec))
print(f"{MODEL} is {state}")
if state == "RUNNING":
    raise SystemExit(f"{MODEL} is claimed by another machine; pick a different model.")

# Claim it, so no other machine starts the same model.
subprocess.run(
    [sys.executable, "run_claim.py", "--model", MODEL, "--note", "notebook 00"],
    cwd=ROOT,
    check=True,
)

command = [
    sys.executable, "run_train.py",
    "--config", spec["config"],
    "--env", "local",
    "--model", MODEL,
]
if (ROOT / spec["dir"] / "checkpoints" / f"{MODEL}_last.pt").is_file():
    command.append("--resume")
    print("Checkpoint found; resuming from the last completed epoch.")

try:
    subprocess.run(command, cwd=ROOT, check=True)
    subprocess.run(
        [
            sys.executable, "run_evaluate.py",
            "--config", spec["config"],
            "--env", "local",
            "--model", MODEL,
        ],
        cwd=ROOT,
        check=True,
    )
finally:
    subprocess.run(
        [sys.executable, "run_claim.py", "--model", MODEL, "--release"],
        cwd=ROOT,
        check=False,
    )

## 4 · Watch a run in progress

Safe to re-run at any time, including while another process is training: it only
reads the history CSV that the trainer appends to after every epoch.

In [ ]:
WATCH = "resnet50"   # the model to inspect

spec = MODELS[WATCH]
history_path = ROOT / spec["dir"] / "metrics" / f"history_{WATCH}.csv"

if not history_path.is_file():
    print(f"No epochs recorded yet for {WATCH}. The first epoch writes this file.")
else:
    history = pd.read_csv(history_path)
    done, target = len(history), spec["epochs"]
    mean_seconds = history["seconds"].mean()
    best = history.loc[history["val_macro_f1"].idxmax()]
    print(f"{WATCH}: {done}/{target} epochs, {mean_seconds / 60:.1f} min per epoch")
    print(
        f"Best validation macro-F1 so far: {best['val_macro_f1']:.4f} "
        f"(epoch {int(best['epoch'])})"
    )
    if done < target:
        print(f"Estimated time remaining: {(target - done) * mean_seconds / 3600:.1f} h")
    display(history.tail(10))

    axes = history.plot(
        x="epoch", y=["train_loss", "val_loss"], figsize=(11, 3.4),
        marker="o", markersize=3, title=f"{WATCH}: loss per epoch",
    )
    axes.set_ylabel("loss")
    axes.grid(alpha=0.3)

    axes = history.plot(
        x="epoch", y="val_macro_f1", figsize=(11, 3.4), marker="o", markersize=3,
        color="#59a14f", title=f"{WATCH}: validation macro-F1 per epoch",
    )
    axes.set_ylabel("macro-F1")
    axes.grid(alpha=0.3)

## 5 · The comparison, as far as the results go

Builds every cross-model table and figure from the recorded runs. Models that
have not finished are left out with a warning, so this is useful from the first
completed model onward — the figures fill in as the runs land.

The full analysis, with confusion matrices and error examples, is notebook 06.

In [ ]:
from run_compare import build_comparison

comparison = build_comparison(list(MODELS), ROOT / "outputs" / "comparison")
comparison

In [ ]:
from IPython.display import Image, display

figures = ROOT / "outputs" / "comparison" / "figures"
for name in (
    "metric_comparison.png",
    "per_class_comparison.png",
    "history_comparison.png",
    "accuracy_against_cost.png",
):
    path = figures / name
    if path.is_file():
        print(name)
        display(Image(str(path)))
    else:
        print(f"{name}: not available yet (needs more finished runs)")

## 6 · Which model wins which class

The table the discussion section needs: an aggregate score says which model is
best overall, this says what each model is best *at*. It becomes meaningful once
two or more models have been evaluated.

In [ ]:
strengths_path = ROOT / "outputs" / "comparison" / "class_strengths.csv"
if strengths_path.is_file():
    strengths = pd.read_csv(strengths_path)
    if strengths["models_compared"].max() < 2:
        print("Only one model is evaluated so far, so every class is won by default.")
    display(strengths)
else:
    print("Run section 5 first.")